# 🧠 NLA Hallucination Detection Pipeline
**Senior Project — SIIT Thammasat University**  
**Model:** Qwen2.5-7B-Instruct | **Dataset:** HaluEval QA | **NLA Layer:** 20

| Phase | Description | VRAM |
|-------|-------------|------|
| 0 — Setup | Install deps + download models | Low |
| Session 1 | Extract Layer 20 activations (Qwen only) | ~14 GB |
| Session 2 | NLA Verbalization (AV) + AR Scoring | ~40 GB |

> ⚠️ **Session 1 and Session 2 must run in separate kernels** (Qwen + SGLang > 48 GB combined)  
> **Dual-layer strategy:** Layer 8 → linear probe | Layer 20 → NLA verbalization

## 🔧 0. Environment Check

In [ ]:
import torch

print(f'PyTorch:    {torch.__version__}')
print(f'CUDA:       {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU:        {props.name}')
    print(f'Total VRAM: {props.total_memory / 1e9:.1f} GB')
    print(f'Used  VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
!nvidia-smi | grep MiB

## 📦 1. Install Dependencies

In [ ]:
!pip install huggingface_hub accelerate datasets scikit-learn tqdm -q
!pip install "sglang[all]==0.5.6" -q
!cd /workspace/natural_language_autoencoders && pip install -e . -q
print('✅ All dependencies installed!')

## 📥 2. Download Models

In [ ]:
import os
os.environ.update({
    'HF_HOME': '/workspace/hf_cache',
    'TRANSFORMERS_CACHE': '/workspace/hf_cache',
})

from huggingface_hub import snapshot_download

MODELS = {
    'Qwen/Qwen2.5-7B-Instruct':    '/workspace/models/qwen2.5-7b-instruct',
    'kitft/nla-qwen2.5-7b-L20-av': '/workspace/models/nla-av',
    'kitft/nla-qwen2.5-7b-L20-ar': '/workspace/models/nla-ar',
}

for repo_id, local_dir in MODELS.items():
    if os.path.exists(local_dir) and os.listdir(local_dir):
        print(f'⏩ Skip (exists): {local_dir}')
        continue
    print(f'📥 Downloading {repo_id}...')
    snapshot_download(repo_id=repo_id, local_dir=local_dir)
    print(f'✅ Done: {local_dir}')

# Verify
print()
for path in MODELS.values():
    n = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f'  {"✅" if n > 0 else "❌"} {path.split("/")[-1]:35s} ({n} files)')
print('\n✅ All models ready!')

---
## 🧠 SESSION 1 — Extract Layer 20 Activations

> **After running all cells in this section: restart the kernel before Session 2!**

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

# ─── Config ───────────────────────────────────────────
MODEL_PATH = '/workspace/models/qwen2.5-7b-instruct'
LAYER_IDX  = 20
N_SAMPLES  = 200
SAVE_DIR   = '/workspace'

# ─── Load Qwen ────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model.eval()
print(f'✅ Qwen loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ─── Load HaluEval ────────────────────────────────────
dataset = load_dataset('pminervini/HaluEval', 'qa_samples', split='data')
samples = dataset.select(range(N_SAMPLES))
print(f'✅ HaluEval: {len(samples)} samples')

In [ ]:
activations_list, labels_list = [], []

def hook_fn(module, input, output):
    hidden = output[0] if isinstance(output, tuple) else output
    h = hidden[:, -1, :].float()          # last token
    h = h / h.norm(dim=-1, keepdim=True)  # L2-normalize
    activations_list.append(h.detach().cpu())

hook = model.model.layers[LAYER_IDX].register_forward_hook(hook_fn)

for sample in tqdm(samples, desc=f'Layer {LAYER_IDX} extraction'):
    prompt = f"Question: {sample['question']}\nAnswer: {sample['answer']}"
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=512
    ).to('cuda')
    with torch.no_grad():
        model(**inputs)
    labels_list.append(1 if sample['hallucination'] == 'yes' else 0)

hook.remove()

# ─── Save ─────────────────────────────────────────────
activations_np = torch.cat(activations_list, dim=0).numpy()
labels_np      = np.array(labels_list)

np.save(f'{SAVE_DIR}/activations_L20.npy', activations_np)
np.save(f'{SAVE_DIR}/labels_L20.npy',      labels_np)

print(f'\n✅ Saved: {activations_np.shape} | dtype: {activations_np.dtype}')
print(f'   Hallucinated: {labels_np.sum()} | Truthful: {(labels_np==0).sum()}')

In [ ]:
import gc

del model
torch.cuda.empty_cache()
gc.collect()

print(f'✅ Qwen freed | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print('\n⚠️  Restart the kernel now before running Session 2!')

---
## 💬 SESSION 2 — NLA Verbalization + AR Scoring

### ⚠️ Prerequisites
1. Kernel must be **fresh** (Qwen not loaded)
2. SGLang AV server must be running:
```bash
python -m sglang.launch_server \
    --model /workspace/models/nla-av \
    --port 30000 \
    --mem-fraction-static 0.85 \
    --disable-radix-cache \
    --trust-remote-code
```
Wait for server ready message before continuing.

In [ ]:
import sys, torch, numpy as np
sys.path.insert(0, '/workspace/natural_language_autoencoders')
from nla_inference import NLAClient, NLACritic

# ─── Load saved activations ───────────────────────────
activations = np.load('/workspace/activations_L20.npy')
labels      = np.load('/workspace/labels_L20.npy')
print(f'✅ Activations: {activations.shape} | Labels: {labels.shape}')
print(f'   Hallucinated: {labels.sum()} | Truthful: {(labels==0).sum()}')

# ─── Init NLA clients ─────────────────────────────────
client = NLAClient(
    checkpoint_dir='/workspace/models/nla-av',
    sglang_url='http://127.0.0.1:30000'
)
critic = NLACritic(
    checkpoint_dir='/workspace/models/nla-ar',
    device='cuda',
    dtype=torch.bfloat16
)
print(f'\n✅ NLAClient + NLACritic initialized | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ─── Sanity check: single sample ─────────────────────
v = activations[0]
explanation = client.generate(v)
mse, cos    = critic.score(explanation, v)
print(f'\n--- Single-sample test (idx=0) ---')
print(f'Verbalization:\n{explanation}')
print(f'MSE: {mse:.4f} | Cosine: {cos:.4f}')

In [ ]:
from tqdm import tqdm

# ─── Full verbalization (AV) ──────────────────────────
all_explanations = []
for v in tqdm(activations, desc='Verbalizing (AV)'):
    all_explanations.append(client.generate(v))

np.save('/workspace/explanations.npy', np.array(all_explanations, dtype=object))
print(f'✅ Saved {len(all_explanations)} explanations → /workspace/explanations.npy')

In [ ]:
from tqdm import tqdm

# Load from disk if not in memory
if 'all_explanations' not in dir():
    all_explanations = np.load('/workspace/explanations.npy', allow_pickle=True).tolist()
    print(f'✅ Loaded {len(all_explanations)} explanations from disk')

# ─── AR Reconstruction ────────────────────────────────
pred_vecs = []
for exp in tqdm(all_explanations, desc='Reconstructing (AR)'):
    pred_vecs.append(critic.reconstruct(exp))

pred_vecs = torch.stack(pred_vecs)  # [N, hidden_dim]

# ─── Compute metrics ──────────────────────────────────
mse_scale = critic.mse_scale
gold   = torch.tensor(activations, dtype=torch.float32)
gold_n = gold   / gold.norm(dim=-1, keepdim=True)   * mse_scale
pred_n = pred_vecs / pred_vecs.norm(dim=-1, keepdim=True) * mse_scale

mu      = gold_n.mean(dim=0)
fve     = 1 - ((pred_n - gold_n)**2).mean() / ((gold_n - mu)**2).mean()
cos_sim = torch.nn.functional.cosine_similarity(pred_n, gold_n, dim=-1)

# ─── Results ──────────────────────────────────────────
print(f'\n{"="*48}')
print(f'  fve_nrm  : {fve.item():.4f}   (in-dist target: ~0.752)')
print(f'  Mean cos : {cos_sim.mean().item():.4f}   (in-dist target: ~0.890)')
print(f'  Mean MSE : {(2*(1-cos_sim)).mean().item():.4f}   (in-dist target: ~0.200)')
print(f'{"="*48}')
print('\n[Methodological note]')
print('HaluEval is OOD vs NLA training data (WildChat + Ultra-FineWeb).')
print('Lower metrics than paper targets are expected and documented.')

---
## 📋 Next Steps
1. **Semantic theme analysis** — compare verbalization themes: hallucinated vs. truthful
2. **Cluster + summarize** verbalization patterns by label
3. **Download results** from `/workspace/` → terminate pod

Key saved files:
- `/workspace/activations_L20.npy` — Layer 20 activations [200, 3584]
- `/workspace/labels_L20.npy` — Ground-truth labels [200]
- `/workspace/explanations.npy` — AV verbalizations [200]